<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [6]</a>'.</span>

---
title: GRPO for Math Reasoning
sidebarTitle: GRPO Math Reasoning
description: Fine-tune Qwen2.5-1.5B-Instruct with GRPO and granular format rewards to elicit chain-of-thought mathematical reasoning.
tag: "Draft"
---

This lab is based on William Brown's [GRPO demo gist](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) and the accompanying paper:

> Brown, W. (2025). *Granular Format Rewards for Eliciting Mathematical Reasoning Capabilities in Small Language Models.* [GitHub Gist](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb). See also [verifiers](https://github.com/willccbb/verifiers) for ongoing developments.

The script fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` (or `meta-llama/Llama-3.2-1B-Instruct`) on the GSM8K dataset using GRPO via TRL, rewarding the model for both correct answers **and** correctly formatted XML chain-of-thought reasoning traces.

<iframe
  width="100%"
  height="480"
  src="https://www.youtube.com/embed/JIsgyk0Paic"
  title="GRPO for Math Reasoning"
  frameBorder="0"
  allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share"
  allowFullScreen
></iframe>

## Reward design

The key insight is a **granular reward stack** that shapes the output format incrementally:

| Reward function | Signal | Weight |
|---|---|---|
| `correctness_reward_func` | Extracted answer matches ground truth | 2.0 |
| `int_reward_func` | Answer is a valid integer | 0.5 |
| `strict_format_reward_func` | Exact `<reasoning>…</reasoning><answer>…</answer>` pattern | 0.5 |
| `soft_format_reward_func` | Relaxed tag presence check | 0.5 |
| `xmlcount_reward_func` | Partial credit per correct XML tag | 0–0.5 |

The model is prompted to always respond in:

```xml
<reasoning>
...
</reasoning>
<answer>
...
</answer>
```

## Setup

<Note>
This notebook requires **≈ 48 GB of VRAM** at the config below. Reduce `per_device_train_batch_size`, `num_generations`, or `max_completion_length` to fit smaller GPUs (memory scales roughly linearly with `per_device × num_generations × max_completion_length`).
</Note>

Requires a GPU with bfloat16 support. The config below targets a single 96GB GPU (RTX PRO 6000 Blackwell) using willccbb's reference batch shape (`per_device=1`, `grad_accum=4`, `num_generations=16`) — 4 prompts × 16 rollouts per step. `num_generations` is the single biggest lever for GRPO convergence since the advantage estimator is very noisy with fewer samples per group.

In [ ]:
# !pip install torch transformers trl peft datasets wandb
# !pip install flash-attn --no-build-isolation

In [ ]:
import os
# Route W&B logs to the eng-ai-agents project
os.environ["WANDB_PROJECT"] = "eng-ai-agents"

# train_grpo.py
# https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb
# See https://github.com/willccbb/verifiers for ongoing developments

import re
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

# Workaround for editable TRL install (pip install -e .) leaving no dist-info
# metadata in the site-packages tree. Without this, the first checkpoint save
# raises PackageNotFoundError when TRL's generate_model_card calls
# importlib.metadata.version('trl'). Patch `trl.trainer.utils.version` directly
# (a patch on `importlib.metadata.version` alone does not work because TRL's
# utils module imports `version` into its own namespace at import time).
import importlib.metadata as _meta
import trl.trainer.utils as _trlutils
_orig_version = _trlutils.version
def _patched_version(name):
    try:
        return _orig_version(name)
    except _meta.PackageNotFoundError:
        if name == "trl":
            return "0.0.0+editable"
        raise
_trlutils.version = _patched_version

SYSTEM_PROMPT = """
Respond in the following format:

<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

In [ ]:
def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip().replace(",", "").replace("$", "")

def get_gsm8k_questions(split="train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split]
    data = data.map(lambda x: {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    })
    return data

dataset = get_gsm8k_questions()

In [ ]:
# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, flags=re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, flags=re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
# model_name = "meta-llama/Llama-3.2-1B-Instruct"

output_dir = "outputs/Qwen-1.5B-GRPO"
run_name = "Qwen-1.5B-GRPO-gsm8k"

# Single-GPU adaptation of willccbb's 4-GPU config.
# TRL's GRPOTrainer requires `generation_batch_size = per_device × grad_accum × num_processes`
# to be divisible by `num_generations`. On a single GPU (num_processes=1) we achieve
# willccbb's reference "4 prompts × 16 rollouts per step" with:
#     per_device=16 × grad_accum=4 × 1 = 64  →  64 / num_generations=16 = 4 prompts/step.
# num_generations=16 is the key convergence lever (groups too small make the
# (r - mean)/std advantage estimator noisy). The 96GB Blackwell has headroom.
#
# save_strategy="no" skips checkpointing. The editable `pip install -e .[notebooks]`
# inside the container leaves TRL's importlib metadata missing, which crashes the
# model-card writer on checkpoint save. Skipping saves is fine for this demo —
# the trainer's final state (log_history, metrics) is what we need.
training_args = GRPOConfig(
    output_dir=output_dir,
    run_name=run_name,
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    logging_steps=1,
    bf16=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    num_generations=16,
    max_completion_length=786,
    num_train_epochs=1,
    max_steps=300,
    save_strategy="no",
    max_grad_norm=0.1,
    report_to="wandb",
    log_on_each_node=False,
)

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    device_map=None
).to("cuda")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args=training_args,
    train_dataset=dataset,
)

# No-op the model-card writer. TRL's default create_model_card calls
# importlib.metadata.version("trl"), which raises PackageNotFoundError on
# editable installs (the container's `pip install -e .[notebooks]` leaves no
# dist-info). The model card itself is not needed for this demo — we only care
# about trainer.state.log_history for the plots below.
trainer.create_model_card = lambda *args, **kwargs: None

trainer.train()

## Training metrics

The cell below plots the four signals that best tell the story of a GRPO run, pulled from `trainer.state.log_history`:

1. **Total reward** — the scalar that GRPO is actually maximising. Should trend upward; the dashed line is an EMA to cut through the per-step noise that is inherent to on-policy RL.
2. **Per-reward breakdown** — shows *which* reward functions are providing signal. Format rewards (`xmlcount`, `soft_format`, `strict_format`) typically saturate in the first 20–50 steps. `correctness_reward_func` is the hard signal and climbs later, usually after 100+ steps.
3. **KL divergence** — tracks how far the policy has drifted from the reference. A slow, bounded rise is healthy; a spike indicates the policy is collapsing toward a narrow mode and reward hacking is likely.
4. **Average completion length** — reasoning traces tend to *lengthen* as the model learns to think before answering. A sudden collapse to short completions usually means the model found a format shortcut.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Pull the per-step log from the trainer.
log_df = pd.DataFrame(trainer.state.log_history)
# Drop non-training rows (final eval, train_runtime summary, etc.)
log_df = log_df[log_df["step"].notna()].reset_index(drop=True)

def ema(series, alpha=0.1):
    return series.ewm(alpha=alpha, adjust=False).mean()

# TRL key-name discovery (names drift across versions; pick whichever is present)
kl_col = next((c for c in ("kl", "train/kl") if c in log_df), None)
len_col = next((c for c in ("completions/mean_length", "completion_length") if c in log_df), None)
reward_col = next((c for c in ("reward", "train/reward") if c in log_df), None)
reward_std_col = next((c for c in ("reward_std", "train/reward_std") if c in log_df), None)
per_reward_mean = sorted(c for c in log_df.columns if c.startswith("rewards/") and c.endswith("/mean"))
per_reward_std = sorted(c for c in log_df.columns if c.startswith("rewards/") and c.endswith("/std"))

palette = ["#2563eb", "#10b981", "#f59e0b", "#ef4444", "#8b5cf6"]

fig, axes = plt.subplots(3, 2, figsize=(14, 13))

# (0,0) Total reward mean
ax = axes[0, 0]
if reward_col is not None:
    ax.plot(log_df["step"], log_df[reward_col], color="#cbd5e1", linewidth=1, label="per-step")
    ax.plot(log_df["step"], ema(log_df[reward_col]), color="#2563eb", linewidth=2, linestyle="--", label="EMA")
    ax.legend(loc="lower right")
ax.set_title("Total reward — mean")
ax.set_xlabel("step"); ax.set_ylabel("reward")
ax.grid(True, alpha=0.3)

# (0,1) Total reward std
ax = axes[0, 1]
if reward_std_col is not None:
    ax.plot(log_df["step"], log_df[reward_std_col], color="#cbd5e1", linewidth=1, label="per-step")
    ax.plot(log_df["step"], ema(log_df[reward_std_col]), color="#2563eb", linewidth=2, linestyle="--", label="EMA")
    ax.legend(loc="upper right")
ax.set_title("Total reward — std (in-group spread)")
ax.set_xlabel("step"); ax.set_ylabel("std")
ax.grid(True, alpha=0.3)

# (1,0) Per-reward means
ax = axes[1, 0]
for i, col in enumerate(per_reward_mean):
    label = col.removeprefix("rewards/").removesuffix("/mean").removesuffix("_reward_func")
    ax.plot(log_df["step"], ema(log_df[col]), color=palette[i % len(palette)], linewidth=2, label=label)
ax.set_title("Per-reward-function — mean (EMA)")
ax.set_xlabel("step"); ax.set_ylabel("reward")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)

# (1,1) Per-reward stds
ax = axes[1, 1]
for i, col in enumerate(per_reward_std):
    label = col.removeprefix("rewards/").removesuffix("/std").removesuffix("_reward_func")
    ax.plot(log_df["step"], ema(log_df[col]), color=palette[i % len(palette)], linewidth=2, label=label)
ax.set_title("Per-reward-function — std (EMA)")
ax.set_xlabel("step"); ax.set_ylabel("std")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)

# (2,0) KL divergence
ax = axes[2, 0]
if kl_col is not None:
    ax.plot(log_df["step"], log_df[kl_col], color="#64748b", linewidth=1, label="per-step")
    ax.plot(log_df["step"], ema(log_df[kl_col]), color="#dc2626", linewidth=2, linestyle="--", label="EMA")
    ax.legend(loc="upper left")
ax.set_title("KL divergence from reference policy")
ax.set_xlabel("step"); ax.set_ylabel("KL")
ax.grid(True, alpha=0.3)

# (2,1) Completion length
ax = axes[2, 1]
if len_col is not None:
    ax.plot(log_df["step"], log_df[len_col], color="#cbd5e1", linewidth=1, label="per-step")
    ax.plot(log_df["step"], ema(log_df[len_col]), color="#7c3aed", linewidth=2, linestyle="--", label="EMA")
    ax.legend(loc="lower right")
ax.set_title("Average completion length (tokens)")
ax.set_xlabel("step"); ax.set_ylabel("tokens")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Sanity: dump column names so we can verify which TRL keys were present this run.
print("log_history columns:", sorted(log_df.columns.tolist()))

### Why standard deviations matter

In GRPO the advantage is `Ã = (r − μ) / σ` within each group of rollouts. The std is not just a statistical curiosity — it's **in the denominator of the gradient**.

1. **Zero-std groups contribute zero gradient.** If all `num_generations` rollouts for a prompt get the same reward (all succeed or all fail), σ → 0 and the advantage is ill-defined (TRL clamps it to 0). That step is effectively wasted. TRL logs `frac_reward_zero_std` precisely to diagnose this — if it is high (> ~0.5), your group size is too small, your reward is too coarse, or the task is too easy or too hard.

2. **Per-function std tells you which rewards are still discriminating.** `rewards/<func>/std` near zero means the reward function is saturated (almost always on) or dead (almost never on). Early in training the format rewards have high std — some rollouts nail it, others don't — and then their std collapses as they saturate. The *shift* in std across functions is the leading indicator that `correctness_reward_func` is now doing the work, not the format rewards.

3. **Total `reward_std` acts as an exploration gauge.** A falling total `reward_std` over time means the policy is becoming more deterministic — good for exploitation, dangerous if it happens *before* the correctness reward has climbed (premature collapse / reward hacking).

4. **KL and sampling-ratio std track policy drift.** High std in `sampling/importance_sampling_ratio` means the learner and the rollout policy have drifted apart (common with multi-epoch GRPO or offline rollouts) — a cue to shorten the off-policy horizon.

**TL;DR:** means tell you *whether* training works; stds tell you *whether there is still signal left to learn from* — and in GRPO specifically, a zero-std group is a literal zero-gradient step.

## Further reading

- [verifiers](https://github.com/willccbb/verifiers) — extended framework by the same author
- [TRL GRPOTrainer docs](https://huggingface.co/docs/trl/grpo_trainer)
- [DeepSeek-R1 technical report](https://arxiv.org/abs/2501.12948) — original GRPO application to reasoning